In [83]:
import numpy as np
from collections import Counter

In [84]:
X_train = np.genfromtxt("./DATA/X_train_clean.csv", delimiter=',', skip_header=1, dtype=np.float64)
X_test = np.genfromtxt("./DATA/X_test_clean.csv", delimiter=',', skip_header=1, dtype=np.float64)

y_train = np.genfromtxt("./DATA/y_train.csv", delimiter=',', skip_header=1, dtype=np.float64)
y_test = np.genfromtxt("./DATA/y_test.csv", delimiter=',', skip_header=1, dtype=np.float64)

In [85]:
print("Train shape: ", X_train.shape)
print("Test shape: ", X_test.shape)

Train shape:  (400, 12)
Test shape:  (100, 12)


In [86]:
print(np.unique(y_train))

[0. 1.]


In [87]:
def gini(y):
  counts = np.bincount(y.astype(int))
  probabilities = counts / len(y)

  return 1 - np.sum(probabilities**2)

In [88]:
def split(X, y, feature, threshold):
  left_mask = X[:, feature] <= threshold
  right_mask = X[:, feature] > threshold

  X_left = X[left_mask]
  X_right = X[right_mask]

  y_left = y[left_mask]
  y_right = y[right_mask]

  return X_left, X_right, y_left, y_right

In [89]:
def best_split(X, y):
  best_feature = None
  best_threshold = None
  best_impurity = float("inf")

  n_features = X.shape[1]

  for feature in range(n_features):
    thresholds = np.unique(X[:, feature])

    for threshold in thresholds:

      X_l, X_r, y_l, y_r = split(X, y, feature, threshold)

      if len(y_l) == 0 or len(y_r) == 0:
        continue

      impurity = (
          len(y_l) / len(y) * gini(y_l) +
          len(y_r) / len(y) * gini(y_r)
      )

      if impurity < best_impurity:
        best_impurity = impurity
        best_feature = feature
        best_threshold = threshold

  return best_feature, best_threshold

In [90]:
class Node:
  def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
    self.feature = feature
    self.threshold = threshold

    self.left = left
    self.right = right

    self.value = value

In [91]:
def build_tree(X, y, depth=0, max_depth=5):
  if depth >= max_depth or len(np.unique(y)) == 1:
    leaf_value = Counter(y).most_common(1)[0][0]
    return Node(value=leaf_value)

  feature, threshold = best_split(X, y)

  if feature is None:
    leaf_value = Counter(y).most_common(1)[0][0]
    return Node(value=leaf_value)

  X_l, X_r, y_l, y_r = split(X, y, feature, threshold)

  left_child = build_tree(X_l, y_l, depth+1, max_depth)
  right_child = build_tree(X_r, y_r, depth+1, max_depth)

  return Node(feature, threshold, left_child, right_child)

In [92]:
def predict_sample(x, node):
  if node.value is not None:
    return node.value

  if x[node.feature] <= node.threshold:
    return predict_sample(x, node.left)
  else:
    return predict_sample(x, node.right)

In [93]:
def predict(X, tree):
  predictions = []

  for x in X:
    predictions.append(predict_sample(x, tree))

  return np.array(predictions)

In [94]:
def compute_accuracy(y_true, y_pred):
  return np.mean(y_true == y_pred)

In [95]:
tree = build_tree(X_train, y_train, max_depth=5)

y_pred = predict(X_test, tree)

accuracy = compute_accuracy(y_test, y_pred)

print("Decision Tree Accuracy: ", accuracy)

Decision Tree Accuracy:  1.0


In [96]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

In [97]:
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred)

In [98]:
print("Model Evaluation Metrics")
print("------------------------")

print("Accuracy: ", compute_accuracy(y_test, y_pred))
print("Precision: ", precision)
print("Recall: ", recall)
print("F1 Score: ", f1)
print("ROC AUC: ", auc)

Model Evaluation Metrics
------------------------
Accuracy:  1.0
Precision:  1.0
Recall:  1.0
F1 Score:  1.0
ROC AUC:  1.0
